In [ ]:
"""
Created on Mon Mar 10 2025
@author: Max Van Migem
"""
import numpy as np
import os
import time
import sys
import pickle
from psychopy import parallel, visual, gui, data, event, core, monitors
from psychopy.visual import ShapeStim
from psychopy import logging
from math import fabs
rng = np.random.default_rng(seed=None)
linejitter_arr =(np.ones((30,30,2))- rng.random((30,30,2))*2)/80
# file_name = os.getcwd() + '/stim_jitter.npy'
# np.save(file_name,linejitter_arr)

In [ ]:
def generateStimCoordinates(gridcol, gridrow, jitter):
    """
    Generates coordinates used to draw the stimulus lines
    """
    # Calculate spacing of grid
    x_spacing = 1.0 / (gridcol)
    y_spacing = 1.0 / (gridrow)

    # Create an array to store coordinates: every point (x,y) of the grid for every quadrant (4) 
    coord_array = np.empty(shape = (gridcol,gridrow,2,4),dtype= 'object')
    quadrant_set = [[-1,1],[1,1],[1,-1],[-1,-1]]

    # Generate grid coord per quadrant
    for i,quad in enumerate(quadrant_set):
        # Per row
        for row in range(gridrow):
            # Per column
            for col in range (gridcol):

                grid_x = col * x_spacing  # grid points
                grid_y = row * y_spacing  

                grid_x = grid_x * quad[0]   # quadrant position
                grid_y = grid_y * quad[1]   # quadrant position

                coord_array[col,row,0,i] = grid_x  # Store them in big ass array
                coord_array[col,row,1,i] = grid_y

    # flip the matrices
    coord_array[:,:,0,0] = np.flip(coord_array[:,:,0,0], axis=0) 
    coord_array[:,:,1,0] = np.flip(coord_array[:,:,1,0], axis=0) 

    coord_array[:,:,0,2] = np.flip(coord_array[:,:,0,2], axis=1) 
    coord_array[:,:,1,2] = np.flip(coord_array[:,:,1,2], axis=1) 

    coord_array[:,:,0,3] = np.flip(coord_array[:,:,0,3], axis=(0,1)) 
    coord_array[:,:,1,3] = np.flip(coord_array[:,:,1,3], axis=(0,1)) 

    #  Add jitter
    for i in range(4):
        coord_array[:,:,:,i] =  coord_array[:,:,:,i] + jitter[:gridcol,:gridrow,:]
  
    return coord_array

In [1]:
import numpy as np
def generatePredictionList(tr_block, n_odd, mode = 'rotation'):
    """ 
    Generate pseudo randomized properties of every trial (start quadrant and regular or odd trials)
    """
    if n_odd > tr_block/2:
        raise ValueError("Invalid input parameters")
    if mode == 'angle': 
        n_odd = n_odd*4
        tr_block = tr_block*4
    prediction_arr = np.zeros(tr_block , dtype=int)
    # Set alternating 1's and 0's
    prediction_arr[:n_odd * 2:2] = 1
    # Shuffle the array to randomize the positions of 0's
    np.random.shuffle(prediction_arr)
    # Ensure no consecutive 1's
    while any(prediction_arr[i-2] == 1 and (prediction_arr[i-1] == 1 or prediction_arr[i] == 1) for i in range(tr_block-1)):
        np.random.shuffle(prediction_arr)
    
    return prediction_arr



In [ ]:
def generateBlockStarts(n_blocks,uniq_quadrant,sub_odd):
    """
    Generate a list with the start position for each block depending on the localised quadrant
    """
    # Roll depending on odd direction
    if sub_odd == 'anticlockwise':
        quad_li = np.roll(np.arange(4),-1)
    elif sub_odd == 'clockwise':
        quad_li = np.roll(np.arange(4),1)
    # You have the main quad and the diagonal
    start_quad = quad_li[uniq_quadrant]
    diagon_quad = quad_li[(uniq_quadrant +2)%4]
    # Create the arrays
    half_blocks = int(np.ceil(n_blocks/2))
    prime_quad_arr = np.full(half_blocks,start_quad)
    diagon_quad_arr = np.full(half_blocks,diagon_quad)
    
    section_blocks = np.concatenate((prime_quad_arr,diagon_quad_arr))
    block_list =  section_blocks[:n_blocks]
    
    return block_list

In [ ]:
arr_o = generateBlockStarts(28,7,'angle')

In [6]:
def generateTrialTimings(tr_block,isi_dur, jitter,randng=rng):
    """ 
    Generate a list of timings 4 per trial
    """
    block_tlist= []
    for i in range(tr_block):
        trial_tlist = []
        for p in range(4):
            stim_t = isi_dur + randng.uniform(low =-jitter,high=jitter)
            trial_tlist.append(stim_t)
        block_tlist.append(trial_tlist)
    return block_tlist

NameError: name 'rng' is not defined

In [2]:
prediction_arr = np.zeros(30 , dtype=int)
# Set alternating 1's and 0's
prediction_arr[:7 * 2:2] = 1